[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madhudream/rag_simple/blob/main/colab/2a_bm25.ipynb)

# 3a — What is BM25? Keyword search from scratch

Companion notebook to blog post **3a (What is BM25?)**. Run every cell top to bottom —
each cell is one idea from the post, and the printed numbers are the ones discussed there.

The four ideas (detective rules):
1. **Term Frequency** — count the clues
2. **IDF** — rare clues matter more
3. **Saturation** — repetition stops helping
4. **Length normalization** — short focused docs beat long rambling ones

In [ ]:
# Only needed for the PRODUCTION section at the very end (from-scratch BM25 is pure stdlib)
%pip install -q fastembed

## Setup — tokenizer, corpus, `avgdl`, document frequencies

`avgdl` is the yardstick for "a normal-sized document in this corpus":
`(10 + 8 + 11 + 10 + 8) / 5 = 9.4`.

In [ ]:
import math
from collections import Counter

def tokenize(text):
    return text.lower().split()

corpus = [
    "The cat sat on the warm windowsill in the sun",       # doc 0
    "A dog chased the cat around the yard",                 # doc 1
    "Dogs are loyal and love to play fetch in the park",    # doc 2
    "The park has a pond where ducks swim every morning",   # doc 3
    "She planted tomatoes and basil in her garden",         # doc 4
]

docs = [tokenize(d) for d in corpus]
N = len(docs)                              # 5 documents
avgdl = sum(len(d) for d in docs) / N      # average document length

# document frequency: in how many documents does each word appear?
df = Counter()
for d in docs:
    for term in set(d):
        df[term] += 1

print("doc lengths:", [len(d) for d in docs])
print("avgdl = (10 + 8 + 11 + 10 + 8) / 5 =", avgdl)

## Idea 1 — Term Frequency: count the clues

Note `the` appears twice already — raw counting over-rewards common words. Idea 2 fixes it.

In [ ]:
doc1 = docs[1]                 # "A dog chased the cat around the yard"
print(Counter(doc1))

## Idea 2 — IDF: rare clues matter more

```
IDF(term) = log( 1 + (N − n + 0.5) / (n + 0.5) )
```

Worked by hand for `cat` (n = 2 of N = 5 docs):
`log(1 + 3.5/2.5) = log(2.4) ≈ 0.875`

In [ ]:
def idf(term):
    n = df.get(term, 0)
    return math.log(1 + (N - n + 0.5) / (n + 0.5))

for t in ["cat", "park", "the", "unknown"]:
    print(f"IDF {t!r:10} n={df.get(t,0)}  {idf(t):.3f}")

## Idea 3 — Saturation: stop keyword stuffing

Each extra occurrence adds less. Ceiling = `k1 + 1` = 2.5, never reached.
Hand-check `f = 2`: `2 × 2.5 / (2 + 1.5) = 5 / 3.5 ≈ 1.429`.

In [ ]:
k1 = 1.5
def tf_component(f):        # simplified: length normalization added in Idea 4
    return f * (k1 + 1) / (f + k1)

for f in [1, 2, 3, 5, 10, 50]:
    print(f"f={f:<3} tf_component={tf_component(f):.3f}")

## Idea 4 — Length normalization: don't let long docs cheat

Length penalty = `1 − b + b·dl/avgdl` — read `dl/avgdl` as "how many times a normal
document is this one?" Bigger penalty lands on the bottom of the fraction → smaller score.

In [ ]:
b = 0.75
def tf_component_full(f, dl):
    return f * (k1 + 1) / (f + k1 * (1 - b + b * dl / avgdl))

print(f"avgdl = {avgdl:.1f}")
print("short doc (9 words) :", round(tf_component_full(1, 9), 3))
print("long doc (30 words) :", round(tf_component_full(1, 30), 3))

## BM25 assembled — the composition, line for line

`BM25 = sum over query terms of ( IDF × TermFreqComponent )`

Sanity-check the top `cat` result by hand (8-word doc, one occurrence):
`doc_length_norm = 0.888` → `term_freq_saturation = 1.072` → `0.875 × 1.072 = 0.938` ✓

In [ ]:
def term_score(term, doc, k1=1.5, b=0.75):
    freqs = Counter(doc)
    if term not in freqs:
        return 0.0
    f = freqs[term]                            # Idea 1: term frequency
    dl = len(doc)
    doc_length_norm = 1 - b + b * dl / avgdl                          # Idea 4
    term_freq_saturation = f * (k1 + 1) / (f + k1 * doc_length_norm)  # Idea 3
    return idf(term) * term_freq_saturation    # Idea 2 x (Ideas 3+4)

def bm25_score(query, doc):
    return sum(term_score(t, doc) for t in tokenize(query))

def search(query):
    scored = [(bm25_score(query, d), corpus[i]) for i, d in enumerate(docs)]
    scored.sort(reverse=True, key=lambda x: x[0])
    return scored

for q in ["cat", "dog park", "garden tomatoes"]:
    print(f"\nQuery: {q!r}")
    for score, text in search(q):
        if score > 0:
            print(f"  {score:.3f}  {text}")

## PRODUCTION — FastEmbed's BM25 sparse vectors

You won't hand-roll BM25 in a real system. `SparseTextEmbedding("Qdrant/bm25")` produces
the sparse vector for you (and drops stop-words — watch the non-zero count). Weights won't
exactly match our from-scratch scores (different tokenization/IDF details) — the four ideas
are identical everywhere.

In [ ]:
from fastembed import SparseTextEmbedding

model = SparseTextEmbedding(model_name="Qdrant/bm25")
sample = ["A dog chased the cat around the yard",
          "She planted tomatoes and basil in her garden"]
embs = list(model.embed(sample))

e = embs[0]
print("type:", type(e).__name__)
print("num nonzero terms in doc0:", len(e.indices))
print("sample values:", [round(float(v), 3) for v in e.values[:5]])

## Where BM25 fails

Search `puppy` — BM25 will never find a document that only says `dog`. Words, not meaning.

**Next notebook: `2b_vector_search.ipynb`** — search by meaning, so `puppy` finds `dog`.